In [ ]:
import re
import pandas as pd
import os
from pandas.core.frame import DataFrame

def read_and_parse(file_path: str, company_code: str, pattern_str: str = r'/([^/]+.(sh|cmd))') -> DataFrame:
    pattern = re.compile(pattern_str, re.I)
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith('#'):#bypass comment
                continue
            match = pattern.search(line)
            if match:
                data.append((company_code, match.group(1), os.path.basename(file_path)))
    # final_data = set(data)
    df = pd.DataFrame(data, columns=['company','shell_name','cron_location'])

    return df

### edp.cron & edi.cron中的shells

In [ ]:
from opengrok_util import codescan

df_edp_cron = pd.concat([read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edp.cron.cld5', 'WHL'),
                     read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edp.cron.ialcld5', 'IAL')],
                    ignore_index=True)

df_edi_cron = pd.concat([read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edi.cron.cld5', 'WHL'),
                     read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edi.cron.ialcld5', 'IAL')],
                    ignore_index=True)
df_ttl_cron = pd.concat([df_edp_cron, df_edi_cron], ignore_index=True)
df_ttl_cron



### VSC中內容未包含cssss0003.sh的shells

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

param_cssss0003 = [
    ('full', '-"cssss0003.sh"'),
    ('path', '(+". sh" OR +". cmd") -/Ushell')
]

records = codescan.scan(param_cssss0003, fetch_all=True, url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                        headers={"apikey": os.getenv('OPENGROK_API_KEY')})
if records:
    pattern = re.compile(r'/([^/]+.(sh|cmd))', re.I)
    data = []
    for key in records.keys():
        match = pattern.search(key)
        if match:
            system = re.search(r"^/([^/]+)/", key, re.I).group(1)
            if re.match(r'.*/ial/.*', key, re.I):
                data.append(('IAL', system, key, match.group(1)))
            else:
                data.append(('WHL', system, key, match.group(1)))

df_all_shells = pd.DataFrame(data, columns=['company', 'system', 'svn_path', 'shell_name'])
df_all_shells

### 存在於edp.cron & edi.cron但未包含cssss0003.sh的shells

In [ ]:
import os

df_final_results = pd.merge(df_all_shells.drop_duplicates(subset=['svn_path', 'company', 'shell_name']),
                            df_ttl_cron.drop_duplicates(subset=['company', 'shell_name', 'cron_location'])
                            , on=['company', 'shell_name'])
df_final_results = df_final_results.loc[df_final_results['company'] == 'WHL'].sort_values(by=['company', 'system'])

export_file = os.path.curdir + os.sep + 'scan_result.xlsx'
# df_final_results.groupby('system').size().reset_index(name='Size')

df_final_results.to_excel(export_file, sheet_name='shells', index=False)

df_final_results

### HardCode fax server id/pw

In [ ]:
from opengrok_util import codescan
from dotenv import load_dotenv
import os, platform

load_dotenv(override=True)

param_ftp_qad = [
    ('full', '"faxadmin"'),
    ('path', '-/Ushell')
]
# param_ftp_qad.append(('projects', [proj for proj in ['BKG','CCM','CMR','CRS','CSS','DDS','DGS','ECS','EDI','IHD','LMR','OOC','PAM','QAD','SAS','SRS','SKD','SSM','WAS','WHL','WCS']]))

export_file = "c:\\temp\\hardcode_faxserver_idpw.xlsx" if platform.system() == 'Windows' else os.path.curdir + os.sep + 'scan_result.xlsx'
headers = {"apikey": "2vvq3TJXmT6NzEwTmewpO9ZzMDMMAa02"}
file_path, df = codescan.scan_to_excel(param_ftp_qad, fetch_all=True, export_file=export_file,
                                       url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                       headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df

### HardCode faxtw|faxialtw(ftp or mail2fax function)

In [ ]:
from opengrok_util import codescan
from dotenv import load_dotenv
import os, platform

load_dotenv(override=True)

param_ftp_qad = [
    ('full', '/(faxtw|faxialtw)/ -type:jar -type:javaclass'),
    ('path', '-/Ushell')
]
# param_ftp_qad.append(('projects', [proj for proj in ['BKG','CCM','CMR','CRS','CSS','DDS','DGS','ECS','EDI','IHD','LMR','OOC','PAM','QAD','SAS','SRS','SKD','SSM','WAS','WHL','WCS']]))

export_file = "c:\\temp\\hardcode_faxserver_domain.xlsx" if platform.system() == 'Windows' else os.path.curdir + os.sep + 'scan_result.xlsx'
headers = {"apikey": "2vvq3TJXmT6NzEwTmewpO9ZzMDMMAa02"}
file_path, df = codescan.scan_to_excel(param_ftp_qad, fetch_all=True, export_file=export_file,
                                       url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                       headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df

### HardCode fax server directory path

In [ ]:
from opengrok_util import codescan
from dotenv import load_dotenv
import os, platform

load_dotenv(override=True)

param_ftp_qad = [
    ('full', '/vfax/api'),
    ('path', '-/Ushell')
]
# param_ftp_qad.append(('projects', [proj for proj in ['BKG','CCM','CMR','CRS','CSS','DDS','DGS','ECS','EDI','IHD','LMR','OOC','PAM','QAD','SAS','SRS','SKD','SSM','WAS','WHL','WCS']]))

export_file = "c:\\temp\\hardcode_faxserver_path.xlsx" if platform.system() == 'Windows' else os.path.curdir + os.sep + 'scan_result.xlsx'
headers = {"apikey": "2vvq3TJXmT6NzEwTmewpO9ZzMDMMAa02"}
file_path, df = codescan.scan_to_excel(param_ftp_qad, fetch_all=True, export_file=export_file,
                                       url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                       headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df